In [1]:
import json
import pandas as pd
from pathlib import Path
from lkb.kb._kg_builder import build_kg_records, build_topk_map, build_tree_from_glottolog_fs, write_jsonl
from lkb.kb._phylo_tree import phylo_neighbor_records

In [ ]:
typ_df = pd.read_csv('../data/derived/URIELPlus_Union.csv', index_col=0)
typ_df.index = typ_df.index.astype(str)
langs = typ_df.index.tolist()
allowed = set(langs)

parent, children, level, _ = build_tree_from_glottolog_fs(Path('../glottolog'))

details = {}
n = len(langs)
for i, lang in enumerate(langs, start=1):
    details[lang] = phylo_neighbor_records(
        lang,
        k_neighbors=200,
        parent=parent,
        children=children,
        level=level,
        allowed_nodes=allowed,
    )
    if i % 1000 == 0:
        print(f"processed {i}/{n}")

out_path = Path('../data/derived/genetic_neighbours_detailed.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(details, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {out_path} ({n} languages)")

In [2]:
metadata_df = pd.read_csv('../data/derived/metadata.csv', index_col=0)
metadata_df.index = metadata_df.index.astype(str)
geographic_neighbours = json.loads(Path('../data/derived/geographic_neighbours.json').read_text(encoding="utf-8"))
genetic_neighbours = json.loads(Path('../data/derived/genetic_neighbours.json').read_text(encoding="utf-8"))
gen_detail_path = Path('../data/derived/genetic_neighbours_detailed.json')
genetic_neighbour_details = (
    json.loads(gen_detail_path.read_text(encoding="utf-8")) if gen_detail_path.exists() else {}
)
topk_map = build_topk_map(pd.read_csv('../data/features/topk_per_feature.csv'))
parent, children, level, names = build_tree_from_glottolog_fs(Path('../glottolog'))

In [4]:
nodes, edges = build_kg_records(
    typ_df=typ_df,
    metadata_df=metadata_df,
    geographic_neighbours=geographic_neighbours,
    genetic_neighbours=genetic_neighbours,
    genetic_neighbour_details=genetic_neighbour_details,
    topk_map=topk_map,
    parent=parent,
    children=children,
    level=level,
    names=names,
)

write_jsonl(Path('../artifacts/resources/kg_nodes.jsonl'), nodes)
write_jsonl(Path('../artifacts/resources/kg_edges.jsonl'), edges)
print(f"Wrote {len(nodes)} nodes -> {'../artifacts/resources/kg_nodes.jsonl'}")
print(f"Wrote {len(edges)} edges -> {'../artifacts/resources/kg_edges.jsonl'}")

Wrote 29431 nodes -> ../artifacts/resources/kg_nodes.jsonl
Wrote 2181156 edges -> ../artifacts/resources/kg_edges.jsonl
